## Family History Notes from MIMIC-Note

In [ ]:
import re
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

c = pd.read_csv('MIMIC Data/mimic-iv-note-deidentified-free-text-clinical-notes-2.2/note/discharge.csv')

# Function to extract the Family History section
def extract_family_history(text):
    match = re.search(r'Family History:\n(.*?)\n\n', text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None

# Apply the function to the 'text' column
c['family_history'] = c['text'].apply(extract_family_history)


c = c.dropna(subset = 'family_history').reset_index(drop = True)

# Define a function to clean the notes
def clean_notes(text):
    # Check if 'Physical Exam:' is in the text
    if '\nPhysical Exam:' in text:
        # Split the text at '\nPhysical Exam:' and take the first part
        return text.split('\nPhysical Exam:', 1)[0]
    else:
        # If 'Physical Exam:' is not found, return the original text
        return text

# Apply the function to the 'family_history_notes' column
c['family_history'] = c['family_history'].apply(clean_notes)


# BREAST CANCER

# List of keywords related to breast cancer
breast_cancer_keywords = [
    "breast cancer", "BRCA1", "BRCA2", "TP53", "PTEN", "STK11", 
    "CDH1", "PALB2", "CHEK2", "Li-Fraumeni syndrome", 
    "Cowden syndrome", "Peutz-Jeghers syndrome"
]

# Function to check for breast cancer mentions
def check_breast_cancer(text):
    for keyword in breast_cancer_keywords:
        if keyword.lower() in text.lower():
            return True
    return False

# Apply the function to the 'text' column
c['Breast Cancer'] = c['family_history'].apply(check_breast_cancer)


# OVARIAN CANCER

# List of keywords related to breast cancer
ovarian_cancer_keywords = [
    "ovarian cancer", "BRCA1", "BRCA2", "RAD51C", "RAD51D", "BRIP1",
    "Lynch Syndrome"
]

# Function to check for breast cancer mentions
def check_ovarian_cancer(text):
    for keyword in ovarian_cancer_keywords:
        if keyword.lower() in text.lower():
            return True
    return False

# Apply the function to the 'text' column
c['Ovarian Cancer'] = c['family_history'].apply(check_ovarian_cancer)

# COLORECTAL CANCER

# List of keywords related to breast cancer
colorectal_cancer_keywords = [
    "colorectal cancer", "APC", "MLH1", "MSH2", "MSH6", "PMS2", "EPCAM", "MUTYH",
    "familial adenomatous polyposis", "MUTYH-associated polyposis", "Lynch syndrome"
]

# Function to check for breast cancer mentions
def check_colorectal_cancer(text):
    for keyword in colorectal_cancer_keywords:
        if keyword.lower() in text.lower():
            return True
    return False

# Apply the function to the 'text' column
c['Colorectal Cancer'] = c['family_history'].apply(check_colorectal_cancer)

# MERGING THE RELEVANT NOTES FROM MIMIC
subset_df = c[(c['Breast Cancer'] == True) | 
               (c['Ovarian Cancer'] == True) | 
               (c['Colorectal Cancer'] == True)]


# GETTING STRUCTURED DATA FROM MIMIC
subject_ids = subset_df.subject_id.to_list()

d1 = pd.read_csv('MIMIC Data/mimic-iv-2.2/hosp/admissions.csv')
d1 = d1[d1['subject_id'].isin(subject_ids)]
d1 = d1.reset_index(drop = True)

d3 = pd.read_csv('MIMIC Data/mimic-iv-2.2/hosp/diagnoses_icd.csv')
d3 = d3[d3['subject_id'].isin(subject_ids)]
d3 = d3.reset_index(drop = True)

d4 = pd.read_csv('MIMIC Data/mimic-iv-2.2/hosp/patients.csv')
d4 = d4[d4['subject_id'].isin(subject_ids)]
d4 = d4.reset_index(drop = True)

# Merge all tables on 'subject_id' using outer join to retain all columns
merged_df = d1.merge(d3, on='subject_id', how='outer', suffixes=('', '_d3'))
merged_df = merged_df.merge(d4, on='subject_id', how='outer', suffixes=('', '_d4'))

merged_df.to_csv('merged_df.csv', index = False)

# Helper function to return the first mode if available
def mode_func(series):
    modes = series.mode()
    return modes.iloc[0] if not modes.empty else None

# Group by subject_id and aggregate according to your specifications
summary = merged_df.groupby('subject_id').agg(
    # hadm_id: count of unique hospital admissions (visits)
    total_visits=('hadm_id', 'nunique'),
    
    # Drop columns like admittime, dischtime, deathtime, etc. by simply not including them

    # admission_type: mode
    admission_type=('admission_type', mode_func),
    
    # discharge_location: mode
    discharge_location=('discharge_location', mode_func),
    
    # language: mode
    language=('language', mode_func),
    
    # marital_status: mode
    marital_status=('marital_status', mode_func),
    
    # race: mode
    race=('race', mode_func),
    
    # icd_code: list of all unique items
    icd_code=('icd_code', lambda x: list(x.unique())),
    
    # icd_version: list of all unique items
    icd_version=('icd_version', lambda x: list(x.unique())),
    
    # gender: mode
    gender=('gender', mode_func),
    
    # anchor_age: get minimum and maximum as separate columns
    anchor_age_min=('anchor_age', 'min'),
    anchor_age_max=('anchor_age', 'max'),
    
    # anchor_year: mode
    anchor_year=('anchor_year', mode_func),
    
    # anchor_year_group: mode
    anchor_year_group=('anchor_year_group', mode_func),
    
    # dod: mode
    dod=('dod', mode_func)
).reset_index()

def random_dob_from_age(age, current_year):
    birth_year = current_year - age
    start_date = pd.Timestamp(year=birth_year, month=1, day=1)
    end_date = pd.Timestamp(year=birth_year, month=12, day=31)
    days_range = (end_date - start_date).days
    random_offset = np.random.randint(0, days_range + 1)
    return start_date + pd.Timedelta(days=random_offset)

# Get the current year from the system
current_year = pd.Timestamp.today().year

# Create a new 'dob' column using the current_year
summary['anchor_age_generated_dob'] = summary['anchor_age_max'].apply(lambda age: random_dob_from_age(age, current_year))

# summary = summary.copy()  # Ensure you're working with a copy, not a view
summary['gender_generated_label'] = summary['gender'].replace({'F': 'Female', 'M': 'Male'})
summary['gender_generated_code'] = summary['gender'].replace({'F': 1, 'M': 2})

def assign_religion_code(df, label_col='EHR_RELIGION_LABEL', new_col='EHR_RELIGION_CODE'):
    """
    Map religion labels to their codes and create a new column in the DataFrame.
    
    Parameters:
        df (pd.DataFrame): The DataFrame containing the religion label column.
        label_col (str): The name of the column that contains the religion labels.
        new_col (str): The name of the new column to create with the religion codes.
    
    Returns:
        pd.DataFrame: The DataFrame with the new religion code column.
    """
    # Original mapping from codes to labels
    religion_mapping = {
         51.0: "No Spiritual Preference",
         16.0: "The Church of Jesus Christ of Latter Day Saints",
         63.0: "Did not Ask",
         3.0:  "Catholic",
         9.0:  "Christian",
         31.0: "Agnostic",
         29.0: "No Spiritual Needs",
         8.0:  "Atheist",
         52.0: "Spiritual, not Religious",
         19.0: "Non-Denominational Christian",
         15.0: "Jehovah's Witness",
         57.0: "Inactive LDS",
         18.0: "Muslim",
         6.0:  "Buddhist",
         11.0: "Episcopalian",
         56.0: "Not on List",
         2.0:  "Jewish",
         4.0:  "Baptist",
         50.0: "Declined to Answer",
         40.0: "Pagan",
         22.0: "Presbyterian",
         1.0:  "Lutheran",
         30.0: "Not Verified",
         14.0: "Islamic",
         23.0: "Protestant",
         13.0: "Hindu",
         12.0: "Greek Orthodox",
         85.0: "Nature",
         5.0:  "Christian Scientist",
         49.0: "Wiccan",
         39.0: "New Thought",
         43.0: "Shamanism",
         75.0: "Taoist",
         36.0: "Humanist",
         55.0: "Evangelical",
         64.0: "Do Not Visit",
         17.0: "Methodist",
         28.0: "Unitarian",
         67.0: "Deist",
         58.0: "Did not Indicate",
         46.0: "Unable to Answer"
    }
    
    # Invert the mapping: religion label -> code
    label_to_code = {label: code for code, label in religion_mapping.items()}
    
    # Map the labels in label_col to their codes using the inverted mapping and create new column
    df[new_col] = df[label_col].map(label_to_code)
    return df

# Example usage:
summary = assign_religion_code(summary, label_col='EHR_RELIGION_LABEL', new_col='EHR_RELIGION_CODE')

w = summary[['subject_id','anchor_age_generated_dob',
       'gender_generated_label', 'gender_generated_code', 'EHR_RELIGION_LABEL',
       'EHR_RELIGION_CODE']].copy()

rename_dict = {
    'subject_id': 'EHR_PAT_ID',
    'anchor_age_generated_dob': 'BIRTH_DATE',
    'gender_generated_code': 'EHR_SEX_CODE',
    'gender_generated_label': 'EHR_SEX_LABEL'
}

w.rename(columns=rename_dict, inplace=True)

w['EHR_PAT_ID_CODE_SYSTEM'] = 'http://epic.com/uhealth/clarity/patient.pat_id'
w['EHR_SEX_CODE_SYSTEM'] = 'http://epic.com/uhealth/clarity/zc_sex'
w['EHR_RELIGION_CODE_SYSTEM'] = 'http://epic.com/uhealth/clarity/zc_religion'
w['PREV_MET_CRITERIA_LIST'] = ''

w = w[['EHR_PAT_ID_CODE_SYSTEM', 'EHR_PAT_ID', 'BIRTH_DATE',
       'EHR_SEX_CODE_SYSTEM', 'EHR_SEX_CODE', 'EHR_SEX_LABEL',
       'EHR_RELIGION_CODE_SYSTEM', 'EHR_RELIGION_CODE', 'EHR_RELIGION_LABEL',
       'PREV_MET_CRITERIA_LIST']]

In [ ]:
import re
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [ ]:
c = pd.read_csv('MIMIC Data/mimic-iv-note-deidentified-free-text-clinical-notes-2.2/note/discharge.csv')

In [ ]:
# Function to extract the Family History section
def extract_family_history(text):
    match = re.search(r'Family History:\n(.*?)\n\n', text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None

# Apply the function to the 'text' column
c['family_history'] = c['text'].apply(extract_family_history)

In [ ]:
c = c.dropna(subset = 'family_history').reset_index(drop = True)

# Define a function to clean the notes
def clean_notes(text):
    # Check if 'Physical Exam:' is in the text
    if '\nPhysical Exam:' in text:
        # Split the text at '\nPhysical Exam:' and take the first part
        return text.split('\nPhysical Exam:', 1)[0]
    else:
        # If 'Physical Exam:' is not found, return the original text
        return text

# Apply the function to the 'family_history_notes' column
c['family_history'] = c['family_history'].apply(clean_notes)

### Breast Cancer keywords

In [ ]:
# List of keywords related to breast cancer
breast_cancer_keywords = [
    "breast cancer", "BRCA1", "BRCA2", "TP53", "PTEN", "STK11", 
    "CDH1", "PALB2", "CHEK2", "Li-Fraumeni syndrome", 
    "Cowden syndrome", "Peutz-Jeghers syndrome"
]

# Function to check for breast cancer mentions
def check_breast_cancer(text):
    for keyword in breast_cancer_keywords:
        if keyword.lower() in text.lower():
            return True
    return False

# Apply the function to the 'text' column
c['Breast Cancer'] = c['family_history'].apply(check_breast_cancer)


### Ovarian Cancer keywords

In [ ]:
# List of keywords related to breast cancer
ovarian_cancer_keywords = [
    "ovarian cancer", "BRCA1", "BRCA2", "RAD51C", "RAD51D", "BRIP1",
    "Lynch Syndrome"
]

# Function to check for breast cancer mentions
def check_ovarian_cancer(text):
    for keyword in ovarian_cancer_keywords:
        if keyword.lower() in text.lower():
            return True
    return False

# Apply the function to the 'text' column
c['Ovarian Cancer'] = c['family_history'].apply(check_ovarian_cancer)

### Colorectal Cancer keywords

In [ ]:
# List of keywords related to breast cancer
colorectal_cancer_keywords = [
    "colorectal cancer", "APC", "MLH1", "MSH2", "MSH6", "PMS2", "EPCAM", "MUTYH",
    "familial adenomatous polyposis", "MUTYH-associated polyposis", "Lynch syndrome"
]

# Function to check for breast cancer mentions
def check_colorectal_cancer(text):
    for keyword in colorectal_cancer_keywords:
        if keyword.lower() in text.lower():
            return True
    return False

# Apply the function to the 'text' column
c['Colorectal Cancer'] = c['family_history'].apply(check_colorectal_cancer)

In [ ]:
subset_df = c[(c['Breast Cancer'] == True) | 
               (c['Ovarian Cancer'] == True) | 
               (c['Colorectal Cancer'] == True)]

In [ ]:
print(c['Colorectal Cancer'].value_counts())

In [ ]:
print('Number of notes with Breast Cancer: ', c['Breast Cancer'].value_counts()[1])
print('Number of notes with Breast Cancer: ', c['Ovarian Cancer'].value_counts()[1])
print('Number of notes with Breast Cancer: ', c['Colorectal Cancer'].value_counts()[1])

### Previously validated sample of 300 notes with 279 patients

In [ ]:
e = pd.read_csv('family_history_validated_100_copy_unopened.csv')
e.drop('Unnamed: 0', axis = 1, inplace = True)

### Getting all notes of these patients

In [ ]:
note_ids = e['id'].tolist()
validated_d = c[c['note_id'].isin(note_ids)]
subject_ids = validated_d['subject_id'].tolist()
validated_d_all = c[c['subject_id'].isin(subject_ids)]

print("Total number of patients:", validated_d_all.subject_id.nunique())
print("Total number of notes:", validated_d_all.shape[0])
print("Average number of notes per patient:", f"{validated_d_all.groupby('subject_id')['note_id'].count().mean():.2f}")
validated_d_all.to_csv('validated_d_all.csv', index = False)

In [ ]:
validated_d_all

In [ ]:
c.shape

In [ ]:
subset_df.shape